# FILA-Net — screening of training variants on one fold

Before spending the GPU quota on five new fold trainings, each candidate
change is trained on **one fold** and compared with the existing model of
that fold **on the same held-out observations**, with the official micro PQ
and a paired bootstrap (whole observations resampled).

Variants (`VARIANTS` below):

* `seed2` — the same recipe with another random seed. It measures the
  seed-to-seed noise of one fold, and the **2-model ensemble**
  (`baseline+seed2`, maps averaged exactly as at test time) measures what a
  second model per fold would bring;
* `convnext_small` — a larger pretrained encoder (ConvNeXt-S, ImageNet-22k).

**Inputs (Kaggle):** the competition data and the output of the main
notebook (`fila-net-sessione2`: `checkpoints/fold0_best.pt`,
`artifacts/oof_cache`, `artifacts/filanet_tuned.yaml`). *Accelerator: GPU T4 × 2*,
*Internet on* (pretrained weights). The two trainings run in parallel, one per
GPU; a run that does not finish within the session resumes in the next one
(attach this notebook's output). Its outputs live under `artifacts/screening/`,
which the main notebook never takes as fold checkpoints.

**Honesty rule.** Only the numbers printed by the last cells are measurements;
one fold is ~140 observations, so small differences are within the noise
(the `seed2` row shows how large it is).

In [ ]:
# ---------------------------------------------------------------- settings
import os
from pathlib import Path

COMPETITION_DATA = [*Path("/kaggle/input").glob("filament-segmentation-2026"),
                    *Path("/kaggle/input").glob("*/filament-segmentation-2026")]
MODE = os.environ.get("FILANET_MODE") or ("full" if COMPETITION_DATA else "smoke")
CONFIG = "configs/filanet.yaml"
FOLD = 0
VARIANTS = {                                     # name -> overrides of the training config ("section.key": value)
    "seed2": {"seed": 2027},
    "convnext_small": {"model.encoder": "convnext_small.fb_in22k_ft_in1k"},
}
SESSION_HOURS = 11.0
INSTALL_REQUIREMENTS = MODE == "full"
REPO_URL = "https://github.com/BFtechlab-flow/sole"
assert MODE in ("smoke", "full"), MODE
if MODE == "smoke" and Path("/kaggle/working").exists() and not os.environ.get("FILANET_MODE"):
    raise RuntimeError("On Kaggle the competition data are not attached: Add Input -> Competitions -> "
                       "'Solar Filament Segmentation Challenge 2026' (or set FILANET_MODE=smoke for a code check)")

In [ ]:
import copy, json, shutil, subprocess, sys, time

T_START = time.time()


def hours_left():
    return SESSION_HOURS - (time.time() - T_START) / 3600


def find_repo():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "src" / "engine.py").exists():
            return c
    return None


REPO = find_repo()
if REPO is None:
    REPO = Path("/kaggle/working/sole") if Path("/kaggle").exists() else Path.cwd() / "sole"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if INSTALL_REQUIREMENTS:
    import importlib.util

    MODULE_OF = {"opencv-python-headless": "cv2", "PyYAML": "yaml", "scikit-image": "skimage",
                 "scikit-learn": "sklearn", "Pillow": "PIL"}
    todo = [p for p in (REPO / "requirements.txt").read_text().split()
            if p.split("==")[0] not in ("torch", "torchvision")
            and importlib.util.find_spec(MODULE_OF.get(p.split("==")[0], p.split("==")[0].lower())) is None]
    if todo:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *todo], check=True)

import numpy as np
import pandas
import torch
import yaml

ART = Path(os.environ.get("FILANET_ARTIFACTS", REPO / "artifacts"))
SCREEN = ART / "screening"
SCREEN.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO), PYTHONHASHSEED="0")
try:
    display  # noqa: B018
except NameError:
    display = print


def sh(*args):
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {' '.join(map(str, args))}")
    return "".join(lines)


print("repository:", REPO, "| mode:", MODE, "| GPUs:", torch.cuda.device_count())

## 1. Data, frozen folds and the existing model of the fold

The fold assignment is recovered from the OOF cache of the main notebook (the
assignment its checkpoints were trained with), exactly as there.

In [ ]:
from src.cv import assert_group_isolation, fold_assignments, folds_from_oof_cache, read_fold_file, write_fold_file
from src.data import load_coco_records
from src.sessions import collect_previous_checkpoints, find_checkpoint_dirs, logged_epochs, run_fold_jobs
from src.utils import data_paths, find_data_root, load_yaml

if MODE == "smoke":
    from tools.make_synthetic_dataset import make_dataset

    cfg = load_yaml(REPO / "configs/smoke_cpu.yaml")
    cfg["data"]["root"] = str(make_dataset(ART / "synthetic_magfilo", n_train=8, n_test=3, size=256, seed=0))
else:
    cfg = load_yaml(REPO / CONFIG)
    cfg["data"]["root"] = str(find_data_root(cfg["data"]["root"]))
N_FOLDS = int(cfg["n_folds"])
records = load_coco_records(data_paths(cfg)[1])
INPUTS = ([p for p in Path("/kaggle/input").iterdir() if p.name != "competitions"]
          if MODE == "full" and Path("/kaggle/input").exists() else [])

# frozen folds (committed file > OOF cache layout of the main notebook > computed)
frozen, fold_source = None, None
committed = REPO / "configs" / f"folds_{cfg['cv']['group_mode']}{N_FOLDS}.csv"
caches = [c for p in INPUTS for c in sorted(p.rglob("artifacts/oof_cache")) if "screening" not in c.parts]
if MODE == "full" and committed.exists():
    frozen, fold_source = read_fold_file(records, committed, N_FOLDS), str(committed)
for c in caches:
    if frozen is None:
        found = folds_from_oof_cache(records, c)
        if found is not None:
            frozen, fold_source = found, f"OOF cache layout: {c}"
if frozen is None:
    if MODE == "full":
        raise RuntimeError("attach the output of the main notebook (its OOF cache fixes the fold assignment)")
    frozen, fold_source = fold_assignments(records, N_FOLDS, cfg["cv"]["group_mode"]), "computed"
assert_group_isolation(records, frozen, cfg["cv"]["group_mode"])
FOLD_FILE = ART / "folds.csv"
write_fold_file(records, frozen, FOLD_FILE, source=fold_source)
cfg["cv"]["fold_file"] = str(FOLD_FILE)
RUN_CFG = ART / "run_config.yaml"
RUN_CFG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("folds:", fold_source, "| held-out observations of fold", FOLD, ":",
      len({r["file_name"] for r, f in zip(records, frozen) if f == FOLD}))

# the existing model of the fold, its OOF maps and the calibrated post-processing of the main notebook
BASE_CKPT = BASE_CACHE = TUNED = None
if MODE == "full":
    for d in find_checkpoint_dirs(INPUTS):
        if (d / f"fold{FOLD}_best.pt").exists() and logged_epochs(d / f"fold{FOLD}_log.jsonl") >= int(cfg["train"]["epochs"]):
            BASE_CKPT = d / f"fold{FOLD}_best.pt"
    for c in caches:
        if (c / f"fold{FOLD}").is_dir():
            BASE_CACHE = c
    tuned = [t for p in INPUTS for t in sorted(p.rglob("artifacts/filanet_tuned.yaml"))]
    if tuned:
        tcfg = load_yaml(tuned[-1])
        tcfg.pop("rescoring", None)  # the learned filter was trained on the baseline maps: compare without it
        tcfg["data"] = cfg["data"]
        tcfg["cv"] = cfg["cv"]
        TUNED = ART / "tuned_no_rescoring.yaml"
        TUNED.write_text(yaml.safe_dump(tcfg, sort_keys=False))
    if BASE_CKPT is None or BASE_CACHE is None:
        raise RuntimeError("baseline checkpoint / OOF cache of the fold not found in the inputs")
print("baseline checkpoint:", BASE_CKPT, "| baseline OOF cache:", BASE_CACHE, "| calibrated post-processing:", TUNED)

## 2. Training of the variants (one GPU each)

Every variant is an ordinary `scripts/train.py` run of fold `FOLD` with its
own configuration. In smoke mode the baseline itself is trained too.

In [ ]:
variants = dict(VARIANTS)
if MODE == "smoke":
    variants = {"baseline": {}, **{k: dict(v, **{"train.epochs": 1}) for k, v in variants.items()}}
    variants["baseline"]["train.epochs"] = 1
    variants["convnext_small"]["model.encoder"] = "resnet10t"  # another architecture, small enough for a CPU check


def variant_config(name):
    vcfg = copy.deepcopy(cfg)
    for key, value in variants[name].items():
        node = vcfg
        *path, last = key.split(".")
        for part in path:
            node = node[part]
        node[last] = value
    path = SCREEN / name / "config.yaml"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(vcfg, sort_keys=False))
    return path


def ckpt_dir(name):
    return SCREEN / name / "checkpoints"


def finished(name):
    epochs = int(load_yaml(SCREEN / name / "config.yaml")["train"]["epochs"])
    return (ckpt_dir(name) / f"fold{FOLD}_best.pt").exists() and logged_epochs(ckpt_dir(name) / f"fold{FOLD}_log.jsonl") >= epochs


for name in variants:
    variant_config(name)
    previous = [c for p in INPUTS for c in sorted(p.rglob(f"screening/{name}/checkpoints"))]
    if previous:
        collect_previous_checkpoints(previous, ckpt_dir(name), N_FOLDS)

todo = [n for n in variants if not finished(n)]
n_gpu = torch.cuda.device_count()
gpus = list(range(min(n_gpu, max(1, len(todo))))) or [None]
cpus = os.cpu_count() or 4


def train_args(name, remaining, n_parallel):
    args = ["scripts/train.py", "--config", SCREEN / name / "config.yaml", "--fold", FOLD, "--output", ckpt_dir(name)]
    if MODE == "full":
        args += ["--max-hours", f"{max(remaining - 1.2, 0.1):.2f}"]  # keep time for the OOF maps and the evaluation
    if n_parallel > 1:
        args += ["--workers", max(1, cpus // n_parallel)]
    return args


print("variants to train:", todo, "| GPUs:", gpus)
failures = run_fold_jobs(todo, train_args, gpus, ENV, REPO, hours_left, min_hours=1.5)
if failures:
    raise RuntimeError("; ".join(failures))
for name in variants:
    print(f"{name}: {logged_epochs(ckpt_dir(name) / f'fold{FOLD}_log.jsonl')} epochs"
          + (" (finished)" if finished(name) else " - rerun with this output attached to continue"))

## 3. Out-of-fold maps of the variants and of the 2-model ensembles

The held-out observations of the fold are predicted by each new model, and by
the ensemble *existing model + new model* (maps averaged as at test time).

In [ ]:
ALL_DONE = all(finished(n) for n in variants)
CACHE = SCREEN / "oof"
base_pattern = str(BASE_CKPT).replace(f"fold{FOLD}_", "fold{fold}_") if BASE_CKPT else str(ckpt_dir("baseline") / "fold{fold}_best.pt")
jobs = {}
if MODE == "smoke":
    jobs["baseline"] = [str(ckpt_dir("baseline") / "fold{fold}_best.pt")]
for name in variants:
    if name != "baseline":
        jobs[name] = [str(ckpt_dir(name) / "fold{fold}_best.pt")]
        jobs[f"baseline+{name}"] = [base_pattern, str(ckpt_dir(name) / "fold{fold}_best.pt")]


def cache_complete(job):
    m = CACHE / job / "manifest.json"
    return m.exists() and json.loads(m.read_text())["folds"].get(str(FOLD), {}).get("complete", False)


if ALL_DONE:
    pending = [j for j in jobs if not cache_complete(j)]

    def cache_args(job, remaining, n_parallel):
        return ["scripts/cache_oof.py", "--config", RUN_CFG, "--weights-pattern", *jobs[job], "--folds", FOLD,
                "--cache-dir", CACHE / job, "--max-hours", f"{max(remaining - 0.4, 0.05):.2f}"]

    gpus = list(range(min(n_gpu, max(1, len(pending))))) or [None]
    failures = run_fold_jobs(pending, cache_args, gpus, ENV, REPO, hours_left, min_hours=0.3)
    if failures:
        raise RuntimeError("; ".join(failures))
    if MODE == "full":
        BASE_CACHE_DIR = BASE_CACHE
    else:
        BASE_CACHE_DIR = CACHE / "baseline"
    print("OOF maps complete:", {j: cache_complete(j) for j in jobs})
else:
    print("training not finished: the comparison runs in the next session")

## 4. Comparison on the held-out observations (official micro PQ, paired bootstrap)

Each row is compared with the existing model of the fold on the **same
annotator entries**, with the default post-processing and with the calibrated
one of the main notebook (without the learned filter, which was trained on the
existing model's maps). `ci95` is the 95 % interval of the difference
(observations resampled); `P(>0)` the share of bootstrap draws where the
candidate is better.

In [ ]:
rows = []
if ALL_DONE and all(cache_complete(j) for j in jobs):
    post = {"default": RUN_CFG, **({"calibrated": TUNED} if TUNED else {})}
    caches_by_name = {"baseline": BASE_CACHE_DIR, **{j: CACHE / j for j in jobs if j != "baseline"}}
    EVAL = SCREEN / "eval"
    for pname, pcfg in post.items():
        csv = {}
        for cname, cdir in caches_by_name.items():
            out = EVAL / f"{pname}__{cname.replace('+', '_plus_')}"
            sh("scripts/evaluate_oof.py", "--config", pcfg, "--cache-dir", cdir, "--folds", FOLD,
               "--output", out.with_suffix(".json"), "--records-csv", out.with_suffix(".csv"), "--bootstrap", 200)
            csv[cname] = out.with_suffix(".csv")
        for cname in caches_by_name:
            m = json.loads((EVAL / f"{pname}__{cname.replace('+', '_plus_')}.json").read_text())
            row = {"post-processing": pname, "model": cname, "PQ": round(m["pq"], 4), "SQ": round(m["sq"], 4),
                   "RQ": round(m["rq"], 4), "TP": m["tp"], "FP": m["fp"], "FN": m["fn"]}
            if cname != "baseline":
                cmp = EVAL / f"cmp_{pname}__{cname.replace('+', '_plus_')}.json"
                sh("scripts/compare_oof_experiments.py", "--baseline", csv["baseline"], "--candidate", csv[cname],
                   "--output", cmp, "--iterations", 5000)
                r = json.loads(cmp.read_text())["micro_pq"]
                row.update({"delta": round(r["delta_pq"], 4), "ci95": [round(v, 4) for v in r["ci95"]],
                            "P(>0)": round(r["probability_positive"], 3)})
            rows.append(row)
    table = pandas.DataFrame(rows)
    (SCREEN / "screening_results.json").write_text(json.dumps(rows, indent=2))
    pandas.set_option("display.width", 200)
    print(table.to_string(index=False))
else:
    print("comparison not available yet")
print(f"\nmode={MODE}; total time {(time.time() - T_START) / 60:.1f} min")
if MODE == "smoke":
    print("SMOKE MODE: synthetic data - these numbers say nothing about solar filaments.")